In [12]:
region_name = "us-east-1"

active_bucket = "s3://iceberg-wh-east/"
passive_bucket = "s3://iceberg-wh-west/"
database = "berg"

table_name = "icetabledemo1"

In [13]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
import boto3
import subprocess

sp_conf = SparkConf() 
sp_conf.set("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.warehouse", active_bucket)
sp_conf.set("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
sp_conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
sp_conf.set("spark.hadoop.fs.s3a.aws.credentials.provider","com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

spark = SparkSession.builder \
    .appName("Glue-Iceberg-Integration") \
    .config(conf=sp_conf) \
    .getOrCreate()

In [14]:
def get_dynamo_latest_metadata_info(d, t):
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    key = {'dbtable': f"{d}.{t}"}
    try:
        response = table.get_item(Key = key)
        return response["Item"]["metadatafile"]
    except Exception as e:
        print("Error getting item:", e)

def set_dynamo_with_new_latest_metadata_info(d, t, latest_metadata):
    # Create a DynamoDB resource
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    # Define the item to be inserted/updated
    item = {
        'dbtable': f'{d}.{t}',
        'metadatafile': latest_metadata,
    }
    try:
        response = table.put_item(
        Item=item
        )
        print("Item put successfully:", response)
    except Exception as e:
        print("Error putting item:", e)


def get_metadata_from_table(d, t):
    glue = boto3.client("glue", region_name = region_name)
    table = glue.get_table(DatabaseName=d, Name=t)
    parameters = table["Table"]["Parameters"]
    full_path_metadata_location = parameters["metadata_location"]
    return full_path_metadata_location.split('/')[-1]

def update_metadata_table(d, t, latest_metadata):
    glue = boto3.client("glue", region_name = 'us-east-1')
    table = glue.get_table(DatabaseName=d, Name=t)
    table_input = table["Table"]
    table_input["Parameters"]["metadata_location"] = f"{active_bucket}{database}.db/{table_name}/metadata/{metadata}"
    
    keys_to_remove = ['CreateTime', 'UpdateTime', 'IsRegisteredWithLakeFormation', 'CatalogId', 'DatabaseName', 'CreatedBy', 'VersionId', 'IsMultiDialectView']
    
    for key in keys_to_remove:
        if key in table_input: del table_input[key]

    print(table_input)
    glue.update_table(
        DatabaseName=d,
        TableInput=table_input
    )
    return

In [15]:
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS glue_catalog.{database} 
""")

DataFrame[]

In [16]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS glue_catalog.{database}.{table_name} (
        id INT,
        name STRING
    )
    USING iceberg
""")

DataFrame[]

In [17]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

+--------+
|count(1)|
+--------+
|     300|
+--------+



In [18]:
set_dynamo_with_new_latest_metadata_info(database,table_name,get_metadata_from_table(database,table_name))

Item put successfully: {'ResponseMetadata': {'RequestId': '977NH3NDQG04I9QJEL0HEMQSQ3VV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 03 Aug 2025 18:15:26 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': '977NH3NDQG04I9QJEL0HEMQSQ3VV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2745614147'}, 'RetryAttempts': 0}}


In [19]:
import random
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [(i, f"name_{random.randint(1000, 9999)}") for i in range(100)]

# Step 2: Create DataFrame with schema id(int), name(string)
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False)
])
df = spark.createDataFrame(data, schema)

df.createOrReplaceTempView("temp_table1")

df.show()

spark.sql(f"""
    INSERT INTO glue_catalog.{database}.{table_name}
    SELECT id, name FROM temp_table1
""")

+---+---------+
| id|     name|
+---+---------+
|  0|name_5086|
|  1|name_7387|
|  2|name_5427|
|  3|name_5456|
|  4|name_5954|
|  5|name_1281|
|  6|name_3905|
|  7|name_7207|
|  8|name_9475|
|  9|name_3472|
| 10|name_9746|
| 11|name_4388|
| 12|name_9581|
| 13|name_3577|
| 14|name_6969|
| 15|name_6024|
| 16|name_5344|
| 17|name_9468|
| 18|name_8322|
| 19|name_8325|
+---+---------+
only showing top 20 rows



DataFrame[]

In [20]:
set_dynamo_with_new_latest_metadata_info(database,table_name,get_metadata_from_table(database,table_name))

Item put successfully: {'ResponseMetadata': {'RequestId': 'TAT6E6KKBBM8FPFSA1P1HORD1VVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 03 Aug 2025 18:15:32 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': 'TAT6E6KKBBM8FPFSA1P1HORD1VVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2745614147'}, 'RetryAttempts': 0}}


In [21]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

+--------+
|count(1)|
+--------+
|     400|
+--------+



In [22]:
s3_copy = f"aws s3 sync {active_bucket}{database}.db/{table_name}/metadata/ {active_bucket}{database}.db/{table_name}/metadata-east/"
subprocess.run(f"{s3_copy}", shell=True, capture_output=True, text=True, check=True)

CompletedProcess(args='aws s3 sync s3://iceberg-wh-east/berg.db/icetabledemo1/metadata/ s3://iceberg-wh-east/berg.db/icetabledemo1/metadata-east/', returncode=0, stdout='Completed 4.7 KiB/16.3 KiB (52.3 KiB/s) with 3 file(s) remaining\ncopy: s3://iceberg-wh-east/berg.db/icetabledemo1/metadata/00004-1dde5c26-2b95-4cec-8c6d-9cea3010f866.metadata.json to s3://iceberg-wh-east/berg.db/icetabledemo1/metadata-east/00004-1dde5c26-2b95-4cec-8c6d-9cea3010f866.metadata.json\nCompleted 4.7 KiB/16.3 KiB (52.3 KiB/s) with 2 file(s) remaining\nCompleted 11.8 KiB/16.3 KiB (52.2 KiB/s) with 2 file(s) remaining\ncopy: s3://iceberg-wh-east/berg.db/icetabledemo1/metadata/c08101be-2b2e-4423-bfcc-0ee4459eb308-m0.avro to s3://iceberg-wh-east/berg.db/icetabledemo1/metadata-east/c08101be-2b2e-4423-bfcc-0ee4459eb308-m0.avro\nCompleted 11.8 KiB/16.3 KiB (52.2 KiB/s) with 1 file(s) remaining\nCompleted 16.3 KiB/16.3 KiB (71.3 KiB/s) with 1 file(s) remaining\ncopy: s3://iceberg-wh-east/berg.db/icetabledemo1/metada